# Classical Elo from one game

Estimate classical Elo from **one side of one game** (10% / 50% / 90% band). This notebook only trains; the sample was built locally (header index → up to 4,000 sides per 200-Elo bin, 4 games per player). Stockfish does not run here.

**Data:** Lichess `lichess_db_standard_rated_2017-02.pgn`, Elo 800–2400. See `docs/data.md`.

**Colab setup** (fresh runtime, run top to bottom):

- `MyDrive/DATA/games_sample.parquet`
- `MyDrive/DATA/sample_meta.json`
- `MyDrive/src/` — full `src` package

Models, metrics, and plots write back to `MyDrive/DATA`.

**Search:** quantile pinball for the median; early stopping on macro per-bin MAE. Six-row grid in `src/train_quantile.py` (depth / `min_child_weight` / learning rate). Lowest validation macro MAE wins; ties keep the frozen row. Overall MAE, within-150, and coverage are reported, not used to pick the row. Scale on validation; score test once.

**`ship_ready`:** test coverage ≥ 75% after that scale.


In [ ]:
from pathlib import Path
import sys

if not Path("/content").exists():
    raise RuntimeError(
        "This notebook trains in Google Colab. Open notebooks/train_site.ipynb there "
        "and run every cell from the top."
    )

from google.colab import drive
import subprocess

drive.mount("/content/drive")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pyarrow>=14", "pandas>=2.0", "numpy>=1.24",
    "scikit-learn>=1.3", "scipy>=1.11", "xgboost>=2.0",
    "matplotlib", "joblib",
])
if not Path("/content/drive/MyDrive").exists():
    raise RuntimeError("Drive mount did not expose /content/drive/MyDrive")
print("[setup] Drive mounted")
print("[setup] Expecting games_sample.parquet and sample_meta.json in MyDrive/DATA")
print("[setup] Expecting schema.py in MyDrive/src")


In [ ]:
from pathlib import Path
import sys


def find_repo():
    path = Path("/content/drive/MyDrive/src")
    if not (path / "schema.py").exists():
        raise RuntimeError(
            "Upload the src package to MyDrive/src so schema.py sits at /content/drive/MyDrive/src/schema.py."
        )
    parent = path.parent
    resolved = str(parent.resolve())
    if resolved not in sys.path:
        sys.path.insert(0, resolved)
    print(f"[setup] repo -> {resolved}")
    return parent


def find_data():
    path = Path("/content/drive/MyDrive/DATA")
    if not (path / "games_sample.parquet").exists():
        raise RuntimeError(
            "Upload games_sample.parquet to MyDrive/DATA after python -m src.pipeline finishes locally."
        )
    print(f"[setup] data -> {path}")
    return path


REPO = find_repo()
DATA = find_data()


## Plan

Hypothesis: a one-step change in tree size can lower validation macro MAE without leaving the quantile objective. If it cannot, the frozen row stays.

Sweep: `SEARCH_GRID` in `src/train_quantile.py`. One axis moves at a time off depth 4, `min_child_weight` 10, and learning rate 0.05: `frozen`, `depth-3`, `depth-5`, `min-child-5`, `min-child-20`, `lr-0.1`. Everything else stays at `FROZEN_PARAMS`.

Selection metric: validation macro per-bin MAE of the median. Recorded beside it, and not used to choose: overall MAE, within 150 Elo, and validation coverage after that row's own interval scale.

Baseline: a linear model on the Stockfish columns, scored with the same macro MAE on validation.

Test is not an input to this plan.


In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.metrics import apply_interval, within_tolerance
from src.train_quantile import (
    SEARCH_GRID,
    build_matrices,
    engine_linear_macro,
    feature_report,
    gate_sample,
    predict_quantiles,
    search_validation,
    ship_decision,
    summarize_predictions,
    write_bundle,
)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

print("[plan] selection = lowest validation macro-bin MAE; ties keep the frozen row")
print("[plan] reported, not used to choose: MAE, within 150, coverage")
for spec in SEARCH_GRID:
    print(
        f"[plan] {spec['name']:14s} depth={spec['max_depth']}  "
        f"min_child_weight={spec['min_child_weight']}  lr={spec['learning_rate']}"
    )

games = pd.read_parquet(DATA / "games_sample.parquet")
meta_path = DATA / "sample_meta.json"
sample_meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.exists() else {}
print(f"[load] rows={len(games):,}  cols={games.shape[1]}")
if sample_meta:
    print(f"[load] {sample_meta.get('protocol', '')}")
    print(
        f"[load] sides={sample_meta.get('sides_selected')}  "
        f"engine_games={sample_meta.get('engine_games')}  "
        f"cross_split_dropped={sample_meta.get('dropped_cross_split_sides')}"
    )

splits = gate_sample(games)
for name, frame in splits.items():
    print(
        f"[split] {name:5s}  rows={len(frame):,}  players={frame['player_key'].nunique():,}  "
        f"mean_elo={frame['elo'].mean():.1f}"
    )

bin_counts = (
    games.groupby(["elo_bin", "split"], observed=True)
    .size()
    .unstack(fill_value=0)
)
print("[bins] rows per rating bin")
print(bin_counts.to_string())
thin = sorted(games.loc[games["thin_bin"].astype(bool), "elo_bin"].unique())
print(f"[bins] thin tail bins (kept whole): {thin or 'none'}")

built = build_matrices(splits)
print(f"[fit] features={len(built['feature_names'])}")
y_train = splits["train"]["elo"].to_numpy(dtype=float)
y_val = splits["val"]["elo"].to_numpy(dtype=float)
y_test = splits["test"]["elo"].to_numpy(dtype=float)


## Baseline

Linear regression on the Stockfish block only. Same validation macro MAE as the search. Test rows are not used to fit the line.


In [ ]:
linear_mae = engine_linear_macro(splits["train"], splits["val"])
print(f"[baseline] engine-linear validation macro MAE={linear_mae:.2f}")


## Search

Six quantile trios. Each median early-stops on macro per-bin MAE. The row with the lowest validation macro MAE is kept. Within-150 can be higher on a row that loses.


In [ ]:
found = search_validation(
    built["matrices"]["train"],
    y_train,
    built["matrices"]["val"],
    y_val,
    splits["val"]["elo_bin"].to_numpy(),
    built["feature_names"],
)
models = found["models"]
chosen = found["chosen"]
search_table = pd.DataFrame(found["rows"])
print(search_table.round(3).to_string(index=False))
print(
    f"[search] chose {chosen['name']}  "
    f"val macro MAE={chosen['val_macro_mae']:.2f}  "
    f"engine-linear={linear_mae:.2f}  "
    f"val within 150={100 * chosen['val_within_150']:.1f}%"
)
print("[search] within-150 did not choose the row")


## Test, scored once

The scale below was chosen on the winning row's validation predictions. This cell predicts the held-out games and stops. Re-running it to compare another configuration uses test as a tuning set; run the notebook from the top instead.

`ship_ready` is coverage of at least 75%. A wide bin, among bins with at least 20 rows, has coverage under 70% or a median interval of at least 400 Elo. Within 150 is the share of test rows whose median guess is that close. It is not the fit objective.


In [ ]:
scale = float(chosen["interval_scale"])
print(
    f"[interval] scale={scale:.2f}  "
    f"validation coverage={100 * chosen['val_coverage']:.1f}%"
)

test_q = predict_quantiles(models, built["matrices"]["test"])
test_summary = summarize_predictions(
    y_test,
    splits["test"]["elo_bin"].to_numpy(),
    *test_q,
    scale,
)
decision = ship_decision(test_summary)
low, mid, high = apply_interval(*test_q, scale)
test_within_150 = within_tolerance(y_test, mid, 150.0)
print(
    f"[test] coverage={100 * test_summary['coverage']:.1f}%  "
    f"macro MAE={test_summary['macro_mae']:.1f}  "
    f"MAE={test_summary['mae']:.1f}  "
    f"within 150={100 * test_within_150:.1f}%  "
    f"ship_ready={decision['ship_ready']}"
)
print(f"[test] wide bins: {decision['wide_bins'] or 'none'}")
print(
    f"[test] engine-linear validation macro MAE was {linear_mae:.1f}. "
    "Test was not used to fit that line or to choose the search row."
)
report = pd.DataFrame(test_summary["bins"])
print(report.round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(6.2, 5.4))
ax.scatter(y_test, mid, s=12, alpha=0.35, color="#2a6fbb")
lims = [float(min(y_test.min(), mid.min())), float(max(y_test.max(), mid.max()))]
ax.plot(lims, lims, color="black", lw=1)
ax.set_xlabel("Actual Elo")
ax.set_ylabel("Predicted median Elo")
ax.set_title("Test median vs actual Elo")
fig.tight_layout()
fig.savefig(DATA / "pred_vs_actual.png", bbox_inches="tight")
plt.show()
print("[plot] saved pred_vs_actual.png. Points above the diagonal are guessed too high.")


## What moves the guess

Importance is the rise in validation MAE when that column is shuffled. The Elo curve is the mean actual rating in quantile bins of the feature, also on validation. Test rows are not used to rank features.


In [ ]:
importance, top = feature_report(
    models["q50"],
    built["matrices"]["val"],
    y_val,
    n_repeats=5,
    top_k=5,
)
print("[features] top of the validation permutation ranking")
print(importance.head(12).round(3).to_string(index=False))
print()
for rank, item in enumerate(top, start=1):
    print(f"{rank}. {item['feature']}  (validation MAE rises by {item['importance']:.2f} Elo when shuffled)")
    print(f"   {item['description']}")

show = importance.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.barh(show["feature"], show["importance"], color="#415a77")
ax.set_xlabel("Validation MAE increase (Elo)")
ax.set_ylabel("Feature")
ax.set_title("Permutation importance of the median model")
fig.tight_layout()
fig.savefig(DATA / "feature_importance.png", bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(11, 6.2))
flat = list(axes.ravel())
for axis, item in zip(flat, top):
    curve = item["curve"]
    if not curve:
        axis.set_axis_off()
        axis.set_title(item["feature"])
        continue
    xs = [point["value_mean"] for point in curve]
    ys = [point["elo_mean"] for point in curve]
    axis.plot(xs, ys, marker="o", color="#2a6fbb")
    axis.set_title(item["feature"])
    axis.set_xlabel("Feature value")
    axis.set_ylabel("Mean actual Elo")
flat[-1].set_axis_off()
fig.suptitle("Validation: mean actual Elo across each feature", y=1.02)
fig.tight_layout()
fig.savefig(DATA / "feature_curves.png", bbox_inches="tight")
plt.show()
print("[plot] saved feature_importance.png and feature_curves.png")


## Save

`model_q10.json`, `model_q50.json`, `model_q90.json`, `encoder.joblib`, and `model_card.json` land in `MyDrive/DATA`, so the bundle remains after the runtime ends.

Read `ship_ready` from coverage. Read within-150 as a second description of the same median, not as the reason the row was kept. If the frozen row lost, the validation table shows the margin.


In [ ]:
result = {
    "chosen": chosen["name"],
    "chosen_params": {
        "max_depth": chosen["max_depth"],
        "min_child_weight": chosen["min_child_weight"],
        "learning_rate": chosen["learning_rate"],
    },
    "val_macro_mae": chosen["val_macro_mae"],
    "engine_linear_val_macro_mae": linear_mae,
    "test_coverage": test_summary["coverage"],
    "test_macro_mae": test_summary["macro_mae"],
    "test_mae": test_summary["mae"],
    "test_within_150": test_within_150,
    "ship_ready": decision["ship_ready"],
    "wide_bins": decision["wide_bins"],
}
print("[result]", json.dumps({key: result[key] for key in (
    "chosen", "test_coverage", "test_macro_mae", "test_mae", "test_within_150", "ship_ready", "wide_bins",
)}))

card = {
    "protocol": (
        "Game-level quantiles at 0.1, 0.5, 0.9. "
        "A six-row validation search selects on macro per-bin MAE. "
        "MAE, within-150, and coverage are reported and do not choose the row. "
        "Interval scale is fit on the chosen validation predictions. Test is scored once."
    ),
    "pool": "classical",
    "feature_names": built["feature_names"],
    "numeric_features": built["numeric_features"],
    "search": found["rows"],
    "chosen": result["chosen_params"] | {"name": chosen["name"], "frozen": chosen["frozen"]},
    "interval_scale": scale,
    "val_coverage": chosen["val_coverage"],
    "val_macro_mae": chosen["val_macro_mae"],
    "val_mae": chosen["val_mae"],
    "val_within_150": chosen["val_within_150"],
    "test_coverage": test_summary["coverage"],
    "test_macro_mae": test_summary["macro_mae"],
    "test_mae": test_summary["mae"],
    "test_within_150": test_within_150,
    "test_bins": test_summary["bins"],
    "ship_ready": decision["ship_ready"],
    "wide_bins": decision["wide_bins"],
    "engine_linear_val_macro_mae": linear_mae,
    "top_features": top,
    "watched_features": [item["feature"] for item in top],
    "sample": {
        "sides_selected": sample_meta.get("sides_selected"),
        "engine_games": sample_meta.get("engine_games"),
        "bins": sample_meta.get("bins"),
    },
}
write_bundle(DATA, models, built["encoder"], card, importance=importance)
print(f"[save] ship_ready={decision['ship_ready']}  within_150={100 * test_within_150:.1f}%")
result


## Next steps

- If `ship_ready` is false, change the sample or the features. Do not score test again to pick a different search row.
- Within-150 is on the card so a later experiment can ask whether it should become the objective. This run does not make that change.
- Do not add win rate (`result_mean`) to chase a lower MAE. That column stands in for opponent strength.
